In [ ]:

import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
import nltk as nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import plotly.express as px
from collections import Counter
from nltk.tokenize import word_tokenize
import string

Load Data

In [ ]:
data=pd.read_csv('https://raw.githubusercontent.com/Himanshu-1703/reddit-sentiment-analysis/refs/heads/main/data/reddit.csv')
data.head(5)

In [ ]:
data.shape

In [ ]:
data.info()

In [ ]:
data.sample()["clean_comment"].values

In [ ]:
data.isna().sum()

In [ ]:
data.dropna(inplace=True)

In [ ]:
data.duplicated().sum()

In [ ]:
data.drop_duplicates(inplace=True)

In [ ]:
data.duplicated().sum()

In [ ]:
data[(data["clean_comment"].str.strip()=='')]

In [ ]:
data=data[~(data['clean_comment'].str.strip()=='')] #to remove \n

In [ ]:
#convert the clean-comment colums to lowercase
data["clean_comment"]=data["clean_comment"].str.lower()

data.head()

In [ ]:
# Remove trailing and leading whitespaces from the clean_comment colums
data["clean_comment"]= data["clean_comment"].str.strip()

In [ ]:
#identify comments containing URLs
url_patt = r'http[s]?://\S+'
comment_with_urls = data[data['clean_comment'].str.contains(url_patt, regex=True, na=False)]


In [ ]:
#remove \n from comment
data['clean_comment'] = data['clean_comment'].str.replace('\n', ' ', regex=True)


EDA


In [ ]:
# distribution of classes

sns.countplot(data=data,x="category")

In [ ]:
# frequency distribution of sentiments

data['category'].value_counts(normalize=True).mul(100).round(2)

In [ ]:
#to create a new colum word count
data["word_count"]=data["clean_comment"].apply(lambda x:len(x.split()))

In [ ]:
data.head()

In [ ]:
# distribution of word_count per sentiment

sns.displot(data,x='word_count',col='category',kind='kde')

In [ ]:
 #distribution of word_count per sentiment

sns.kdeplot(data,x='word_count',hue='category',fill=True)

In [ ]:
# plot wordclouds for each sentiment

def plot_wordcloud(target_class):
    text = ' '.join(data.loc[(data['category'] == target_class),"clean_comment"])
    wordcloud = WordCloud(width=800, height=600).generate(text)
    plt.imshow(wordcloud, interpolation='bilinear')
    plt.axis('off')
    plt.title(f'Word Cloud for Target-->{target_class}')
    plt.show()

for sentiment in data['category'].unique().tolist():
    plot_wordcloud(sentiment)

In [ ]:
# boxplots for pos tags for each sentiment


sns.boxplot(data["word_count"])

In [ ]:
data['msg_len'] = data['clean_comment'].apply(lambda x: len(x.split()))

plt.figure(figsize=(10,5))
sns.histplot(data=data, x='msg_len', hue='category', bins=50, kde=True)
plt.title("Distribution of Message Length ( +ve , -ve , N)")
plt.xlabel("Message Length (words)")
plt.ylabel("Frequency")
plt.show()

In [ ]:
# avg word counts among sentiments

sns.barplot(data,x='category',y='word_count',estimator='mean')

In [ ]:
sns.barplot(data,x='category',y='word_count',estimator='median')

In [ ]:

# Download stopwords if not already
nltk.download("stopwords")

# English stopwords
stop_words = set(stopwords.words("english"))


# Function to check if text contains stopwords
def contains_stopwords(text):
    words = re.findall(r'\w+', str(text).lower())
    return any(word in stop_words for word in words)

# Create new column
data["num_stopwords"] =data["clean_comment"].apply(contains_stopwords)

print(data)

In [ ]:

# kdeplot of stopwords count

sns.kdeplot(data,x='num_stopwords',hue='category',fill=True)


In [ ]:
import nltk

nltk.download('punkt')    
nltk.download('punkt_tab')

In [ ]:

# top 15 words and their count in each sentiment

def tokenize_sentences(text):
    tokens = nltk.word_tokenize(text)
    return tokens

tokens = data.loc[:,"clean_comment"].apply(tokenize_sentences)

positive_tokens = tokens.loc[data['category'] == "1"].sum()
neutral_tokens = tokens.loc[data['category'] == "0"].sum()
negative_tokens = tokens.loc[data['category'] == "-1"].sum()

In [ ]:
import nltk, string, re
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')

stopwords_En = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    # lowercase
    text = text.lower()
    # remove punctuation
    text = "".join([char for char in text if char not in string.punctuation])
    # remove numbers
    text = re.sub(r'\d+', '', text)
    # tokenize
    tokens = word_tokenize(text)
    # remove stopwords
    tokens = [word for word in tokens if word not in stopwords_En]
    # lemmatization
    tokens = [lemmatizer.lemmatize(word) for word in tokens]
    return " ".join(tokens)

data['cleaned_text'] = data['clean_comment'].apply(preprocess_text)


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd
#tfidf = TfidfVectorizer(min_df=1)
tfidf = TfidfVectorizer( ngram_range=(2,2))
features_tfidf = tfidf.fit_transform(data['cleaned_text'])
print(features_tfidf.shape)
print('Sparse Matrix :\n', features_tfidf)
features_tfidf = tfidf.fit_transform(data['cleaned_text'])

features_tfidf.columns = tfidf.get_feature_names_out()
features_tfidf

In [ ]:

import numpy as np
import pandas as pd

sample_features = features_tfidf[:, :30].toarray()

features_df = pd.DataFrame(sample_features, columns=tfidf.get_feature_names_out()[:30])


corr = np.corrcoef(features_df.T)

import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(12,8))
sns.heatmap(corr, cmap="coolwarm", 
            xticklabels=tfidf.get_feature_names_out()[:30], 
            yticklabels=tfidf.get_feature_names_out()[:30])
plt.title("Correlation Heatmap of Top 30 TF-IDF Features")
plt.show()